# Sampling vs MCTS: Phase 2 (Colab) — Run All friendly

**Plan A pgx-native pipeline.** Runs end-to-end if you've done the one-time wandb headless setup (see cell 9 markdown). Estimated total wall-clock:

| Stage | Time |
|---|---|
| Setup (cells 1-9) | 5-10 min (one-time per session; auto-skips if cached) |
| Setup smoke (cell 11) | <1 min |
| Per-iter bench cells 13-14 | 2-5 min |
| Mini Phase 2 both arms (cells 16, 17) | ~30-90 min |
| **Full Phase 2 both arms (cells 19, 20)** | **~10-30 h** |

**Recommended GPU**: G4 (RTX PRO 6000 Blackwell, 96 GB) for ~2× A100 throughput at similar unit/h.

**Resume on Colab disconnect**: full Phase 2 cells write to `--ckpt-dir` on Drive. If Colab disconnects mid-run, just re-run that cell — it auto-resumes from `latest.pkl`.

## 1. GPU + Drive

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_BASE = '/content/drive/MyDrive/sampling-chess'
os.makedirs(DRIVE_BASE, exist_ok=True)
os.makedirs(f'{DRIVE_BASE}/checkpoints', exist_ok=True)
print('Drive base:', DRIVE_BASE)

## 2. Clone + install

In [ ]:
%cd /content
![ -d sampling-chess ] || git clone https://github.com/hongttisme/sampling-chess.git
%cd sampling-chess
!git pull --ff-only

In [ ]:
# Editable install + ml extras (jax/flax/optax/mctx/wandb) + pgx separately.
!pip install -q -e ".[ml]" pgx
# Stockfish for eval opponent.
!apt-get install -qq -y stockfish && which stockfish

In [ ]:
# Verify the GPU stack is wired through.
import jax, flax, optax, mctx, pgx
print(f'jax {jax.__version__}  flax {flax.__version__}  optax {optax.__version__}')
print(f'mctx {mctx.__version__}  pgx {pgx.__version__}')
print(f'devices: {jax.devices()}')
print(f'backend: {jax.default_backend()}')

## 3. wandb login

Paste your API key from https://wandb.ai/authorize when prompted.

In [ ]:
# Headless-friendly wandb auth.
# For "Run All and leave overnight" to work without an interactive prompt,
# save your API key ONCE to /content/drive/MyDrive/sampling-chess/wandb_key.txt
# (just the key string, e.g.  echo "abc123..." > .../wandb_key.txt).
# If the file exists this loads it into the env and `wandb login` returns
# immediately. If it's missing, `wandb login` prompts interactively (will
# block Run All until you paste).
import os
KEY_PATH = '/content/drive/MyDrive/sampling-chess/wandb_key.txt'
if os.path.exists(KEY_PATH):
    with open(KEY_PATH) as f:
        os.environ['WANDB_API_KEY'] = f.read().strip()
    print(f'wandb: loaded API key from {KEY_PATH} (headless OK)')
else:
    print(f'wandb: no key file at {KEY_PATH}')
    print('  -> Run All will block on the next cell waiting for paste.')
    print('  -> To make headless: paste your key into that file once.')
!wandb login

## 4. Setup smoke (~1 min)

Same args as the local CPU smoke (117s on a 12-core box). Should run in **seconds** on any GPU. If this errors, fix before going further.

In [ ]:
!python scripts/09_phase2_full_smoke.py --arm b --iters 2 --games-per-iter 2 \
    --train-steps 15 --batch 4 --max-plies 8 --eval-every 1 --eval-games 2

## 5. Per-iter bench (~5-30 min)

Single iter at near-real config; the printed wall-clock per iter is what you multiply by `n_iterations` for a budget. **Read the iter-line wall-clock** to decide whether step 6/7's defaults are sensible.

Rough projection:
- iter time ~5 min  → full 50-iter run ~4 h
- iter time ~15 min → full 50-iter run ~12 h
- iter time ~30 min → full 50-iter run ~25 h

In [ ]:
# Arm B bench — usually slower than Arm A so worth profiling first.
!python scripts/09_phase2_full_smoke.py --arm b --iters 1 --games-per-iter 16 \
    --train-steps 100 --batch 64 --max-plies 200 --eval-every 0 --wandb \
    2>&1 | tee /content/drive/MyDrive/sampling-chess/bench_arm_b.log

In [ ]:
# Arm A bench (mctx).
!python scripts/09_phase2_full_smoke.py --arm a --iters 1 --games-per-iter 16 \
    --train-steps 100 --batch 64 --max-plies 200 --eval-every 0 --wandb \
    2>&1 | tee /content/drive/MyDrive/sampling-chess/bench_arm_a.log

## 6. Mini Phase 2 (both arms, ~30 min - 4 h each)

After step 5 you know roughly what an iter costs. The defaults below assume ~5-15 min/iter on G4. Adjust `--iters`, `--games-per-iter`, `--train-steps` if your bench differs.

**Goal**: see loss curve actually decrease across iterations + first signs of eval Elo above random.

In [ ]:
# Mini Arm B: 10 iters x 32 games with doc-spec 16M-param model.
# --no-resume guarantees fresh start (any existing ckpt in this dir from
# previous tiny-model runs would crash on shape mismatch).
!rm -rf /content/drive/MyDrive/sampling-chess/checkpoints/mini_arm_b/
!python scripts/09_phase2_full_smoke.py --arm b --iters 10 --games-per-iter 32 \
    --train-steps 200 --batch 128 --max-plies 200 \
    --eval-every 2 --eval-games 32 --eval-skill 0 --wandb --no-resume \
    --ckpt-dir /content/drive/MyDrive/sampling-chess/checkpoints/mini_arm_b \
    2>&1 | tee /content/drive/MyDrive/sampling-chess/mini_arm_b.log

In [ ]:
# Mini Arm A: same config as mini Arm B for matched comparison.
!rm -rf /content/drive/MyDrive/sampling-chess/checkpoints/mini_arm_a/
!python scripts/09_phase2_full_smoke.py --arm a --iters 10 --games-per-iter 32 \
    --train-steps 200 --batch 128 --max-plies 200 \
    --eval-every 2 --eval-games 32 --eval-skill 0 --wandb --no-resume \
    --ckpt-dir /content/drive/MyDrive/sampling-chess/checkpoints/mini_arm_a \
    2>&1 | tee /content/drive/MyDrive/sampling-chess/mini_arm_a.log

## 7. Full Phase 2 — Arm B only this round

doc-spec config: 50 iters x 256 games per iter x 1000 train steps x eval every 5 iters at multiple skill levels. Single arm in this round; Arm A deferred (see cell 20 for diagnosis).

Run each arm in **separate sessions** (Colab disconnects every 12 h on Pro / 24 h on Pro+; checkpoints below are written to Drive so re-running the cell auto-resumes from `latest.pkl`).

Estimated wall-clock for Full Arm B on G4 Blackwell: ~15-20 h.

In [ ]:
# Full Arm B: 50 iters x 256 games x 1000 train steps. doc-spec 16M-param model.
# Eval every 5 iters x 64 games per skill -> ~30 Elo CI per checkpoint.
# NO --no-resume so a Colab disconnect mid-run resumes from latest.pkl on
# next session (just re-run this cell).
!python scripts/09_phase2_full_smoke.py --arm b --iters 50 --games-per-iter 256 \
    --train-steps 1000 --batch 1024 --max-plies 400 \
    --eval-every 5 --eval-games 64 --eval-skill 0 --wandb \
    --ckpt-dir /content/drive/MyDrive/sampling-chess/checkpoints/full_arm_b \
    2>&1 | tee /content/drive/MyDrive/sampling-chess/full_arm_b.log

### Full Arm A — DEFERRED THIS ROUND

Mini Arm A revealed a draw-collapse that the temperature fix did not solve: mctx visit counts are inherently sharp even at tau=1, so 30 plies of "sampling" is effectively near-greedy. Both arms in self-play converge on the same lines, hit max_plies cap, and almost every game draws (avg 30 D / 32 games per iter, v_loss flatlined at ~0.05 because z=0 trivially predicted). Running full Arm A in this state would burn ~50 h of GPU time on data with no real Phase 2 dynamics.

**Plan after Full Arm B completes**:
1. Push an Arm A exploration patch (larger Dirichlet noise, or mctx visit-count temperature > 1, or always-sample early iters).
2. Re-run mini Arm A to verify the fix breaks the draw collapse.
3. Then launch the full Arm A run below.

When ready, copy this command into a fresh code cell:

```bash
!python scripts/09_phase2_full_smoke.py --arm a --iters 50 --games-per-iter 256 \
    --train-steps 1000 --batch 1024 --max-plies 400 \
    --eval-every 5 --eval-games 64 --eval-skill 0 --wandb \
    --ckpt-dir /content/drive/MyDrive/sampling-chess/checkpoints/full_arm_a \
    2>&1 | tee /content/drive/MyDrive/sampling-chess/full_arm_a.log
```

(Add the relevant `--dirichlet-fraction X` / `--mctx-temperature X` / `--always-sample` flag once the patch lands.)

## Notes

- **Default model size is doc-spec (~16M params)**. CLI exposes `--n-layers --d-model --n-heads --ffn-dim` to override; pass smaller values for CPU smoke (e.g., `--n-layers 2 --d-model 64 --n-heads 4 --ffn-dim 128 -> 383k params`).
- **Checkpoints**: `--ckpt-dir DIR` saves `iter_NNNNN.pkl` + `latest.pkl` per iter. Resume is automatic if `latest.pkl` exists in the dir; pass `--no-resume` to start fresh. Mini cells use `--no-resume` to avoid resuming from any leftover tiny-model ckpts. Full cells DO NOT pass `--no-resume` so a Colab disconnect mid-run resumes on next launch (just re-run that single cell).
- **Eval**: `--eval-games N --eval-every K --eval-skill S`. Mini uses 32 games every 2 iters at skill 0. Full uses 64 games every 5 iters at skill 0. To eval at multiple skills, run extra cells with different `--eval-skill` values; the script currently takes one skill per invocation.
- **Wandb dashboard**: https://wandb.ai/hongttisme-xiamen-university-malaysia/sampling-chess

Per-iter wall-clock from cells 13-14 is the source of truth for budgeting; don't trust the rough numbers in this notebook.